# 5.2 — Ridge Regression
### Taming Wild Coefficients with L2 Regularisation

---

## The Problem with Linear Regression

Linear Regression has no restrictions on coefficient size. Given correlated features, it panics and assigns wild values — one feature gets +800, another gets −750 — just to fit the training data.

**Why does this happen?**

Imagine predicting Bangalore flat prices with two correlated features: `area_sqft` and `number_of_bedrooms`. Bigger flat = more bedrooms. They move together.

The model needs their **combined contribution** to equal roughly ₹40 per sqft. There are infinite ways to split this:

```
w_area = 40,   w_bedrooms = 0     ← clean, correct
w_area = 800,  w_bedrooms = -750  ← wild, but MSE is almost the same!
w_area = -200, w_bedrooms = 240   ← completely flipped, still similar MSE
```

Think of it like **You and Suresh carrying a 40kg box together**. As long as the total is 40kg, the job gets done — so one of you could push with 800kg force while the other pulls with 760kg. Absurd in real life. But mathematically, MSE doesn't care.

Result:
- Coefficients become **huge, unstable, and uninterpretable**
- Model **overfits** — great on training data, terrible on new data
- Small change in training data → completely different coefficients

---

## The Analogy

Imagine you are a student cramming for an exam. You memorise every single question from last year's paper — word for word. You score 100% on that paper. But when the actual exam has slightly different questions, you're completely lost.

That's **Linear Regression overfitting** — memorised the training data so hard it's useless on new data.

Now imagine your teacher says:

> *"I will penalise you for memorising too much. For every extra fact you cram beyond what's necessary, I'll deduct marks from your final score."*

Suddenly you stop memorising every tiny detail. You focus on the big patterns that actually matter. You **generalise** better.

**That penalty the teacher adds — that is Ridge Regression.**

---

## What Ridge Actually Does

Ridge takes the MSE loss and **adds a penalty term:**

$$\text{Ridge Loss} = \underbrace{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}_{\text{MSE — be accurate}} + \underbrace{\lambda \sum w_i^2}_{\text{Penalty — stay small}}$$

| Part | Meaning |
|------|---------|
| MSE | Same as before — measures how wrong predictions are |
| $\lambda$ (lambda) | A number YOU choose — controls how hard the penalty hits |
| $\sum w_i^2$ | Sum of squares of all coefficients — the bigger the weights, the bigger this |

The model must minimise **both** at once. These two goals pull against each other:
- MSE wants coefficients to be whatever fits training data best (even w=800)
- Penalty wants coefficients to be small

The model finds a **compromise** — that compromise is the shrunk, stable coefficient.

---

## The Tug of War — Concrete Numbers

Say current coefficient w = 100.

**Without Ridge (λ = 0):**
```
Loss = MSE only
Model says: "w=100 gives best predictions, I'll keep it"
```

**With Ridge (λ = 1):**
```
Loss = MSE + 1 × (100²) = MSE + 10,000
That 10,000 penalty is HUGE.
Model says: "this coefficient is costing me too much. Let me reduce it."
```

Model tries w = 50:
```
Loss = MSE (slightly worse) + 1 × (50²) = MSE + 2,500
```
MSE got a little worse — but penalty dropped from 10,000 to 2,500. **Net loss is lower.** w=50 wins.

The model keeps reducing w until: *"reducing further would hurt MSE more than it saves in penalty."* That's the sweet spot.

---

## What λ (alpha) Controls

> In sklearn, λ is called **`alpha`** because `lambda` is a reserved Python keyword.

| α (alpha) | Penalty strength | Effect |
|-----------|-----------------|--------|
| 0 | None | Pure Linear Regression — can overfit |
| Small (0.01) | Tiny | Coefficients shrink slightly |
| Moderate (1.0) | Balanced | Usually best — good generalisation |
| Large (100) | Heavy | Coefficients shrink aggressively |
| Very large (10,000) | Extreme | All coefficients near zero — underfits |

**α is a dial.** Turn it up → simpler model. Turn it down → more complex model.

---

## Why Does Ridge Never Reach Exactly Zero?

The penalty for one coefficient w is: **penalty = w²**

This is a U-shape (parabola). The "push" Ridge applies at any point = the **slope** of this curve:

$$\text{slope of } w^2 = 2w$$

| w value | Push toward zero (= 2w) |
|---------|------------------------|
| 100 | 200 — huge push |
| 10 | 20 — medium push |
| 1 | 2 — small push |
| 0.001 | 0.002 — tiny push |
| 0 | **0 — zero push** |

The push **disappears exactly when w reaches zero.** It's like a magnet that loses strength as you approach it. You get infinitely close but never quite arrive.

This is also why Ridge **keeps all features** — it shrinks them toward zero but never eliminates them. For feature elimination, you need Lasso (5.3).

---

## The Gradient Descent View

During training, Ridge modifies the update rule:

**Linear Regression update:**
$$w = w - \alpha_{lr} \cdot \frac{\partial MSE}{\partial w}$$

**Ridge update (extra term added):**
$$w = w - \alpha_{lr} \cdot \left(\frac{\partial MSE}{\partial w} + \lambda \cdot 2w\right)$$

That extra **λ · 2w** term subtracts a little from w at **every single step** — proportional to how large w currently is.

- Large w → large subtraction → fast shrinking
- Small w → small subtraction → slow shrinking
- w = 0 → zero subtraction → stops shrinking

Same story as the parabola slope — consistent, elegant, never reaches zero.

---

## Why Scale Before Ridge?

The penalty hits **all coefficients equally** — λΣwᵢ².

But if `area_sqft` ranges from 400–2500 and `distance_km` ranges from 0.5–15, their raw coefficient sizes are on completely different scales.

- area coefficient might be 0.04 (small number, big feature range)
- distance coefficient might be 1.5 (bigger number, smaller feature range)

Ridge would penalise the distance coefficient 37x harder than area — **unfair and wrong.**

**StandardScaler fixes this** — it puts all features on the same scale (mean=0, std=1) before Ridge sees them. Now the penalty is fair across all features.

---

## Ridge vs Linear Regression — Bias-Variance View

| | Linear Regression | Ridge Regression |
|---|---|---|
| **Bias** | Low — no assumptions about coefficient size | Slightly higher — assumes coefficients should be small |
| **Variance** | High — overfits, changes a lot with different data | Lower — stable, generalises better |
| **Coefficients** | Can be wild (±800) | Always calm and reasonable |
| **Features kept** | All | All (just shrunk) |

Ridge deliberately introduces a **small amount of bias** (assuming coefficients should be small) in exchange for a **large drop in variance**. That tradeoff usually wins in the real world.

---

## When to Use Ridge

| Situation | Use Ridge? |
|-----------|----------|
| Many features (50+) | ✅ Yes — multicollinearity almost guaranteed |
| Correlated features | ✅ Yes — Ridge stabilises wild coefficients |
| Linear Regression overfitting | ✅ Yes — Ridge is the fix |
| Want to keep ALL features | ✅ Yes — Ridge never eliminates features |
| Want to eliminate useless features | ❌ No — use Lasso (5.3) instead |

---

## Real World Problem — Bangalore Flat Price with Correlated Features

**Priya's real estate firm** now has a bigger dataset — 50 features including some correlated ones like `area_sqft` and `num_bedrooms`, `floor_number` and `has_lift`, `locality_rating` and `nearby_schools`.

Plain Linear Regression gives wild coefficients on these correlated features. She needs Ridge.

We'll demonstrate:
1. Linear Regression coefficients going wild with correlated features
2. Ridge taming them
3. How α controls the shrinkage
4. Finding the best α with Cross-Validation

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

# WHY these imports:
# Ridge      — sklearn's Ridge implementation (uses optimised solver internally)
# RidgeCV    — Ridge with built-in cross-validation to find best alpha automatically
# Pipeline   — chains scaler + ridge together so scaling never leaks into test data
# StandardScaler — mandatory before Ridge (fair penalty across all features)

np.random.seed(42)

In [ ]:
# ── Step 1: Create Dataset WITH Correlated Features ───────────────────────────
n = 500

# Primary features
area_sqft        = np.random.randint(400, 2500, n)
distance_metro   = np.round(np.random.uniform(0.5, 15, n), 1)
age_years        = np.random.randint(0, 30, n)
locality_rating  = np.random.randint(1, 11, n)
floor_number     = np.random.randint(1, 20, n)

# CORRELATED features — derived from primary ones + small noise
# WHY create correlated features?
# To show what happens when Linear Regression sees two features carrying
# the same information — this is what causes coefficient blow-up
num_bedrooms     = np.round(area_sqft / 500 + np.random.normal(0, 0.3, n)).clip(1, 5)
# num_bedrooms is strongly correlated with area_sqft (bigger flat = more bedrooms)

nearby_schools   = locality_rating + np.random.randint(-1, 2, n)
nearby_schools   = nearby_schools.clip(1, 10)
# nearby_schools is correlated with locality_rating (good locality = good schools)

has_lift         = (floor_number > 5).astype(int)
# has_lift is correlated with floor_number (high floors always have lifts)

# True price formula (only primary features truly matter)
noise = np.random.normal(0, 3, n)
price_lakhs = (
    0.04  * area_sqft +
   -1.5   * distance_metro +
   -0.6   * age_years +
    2.5   * locality_rating +
    0.3   * floor_number +
    10    + noise
)

df = pd.DataFrame({
    'area_sqft':       area_sqft,
    'num_bedrooms':    num_bedrooms,      # correlated with area
    'distance_metro':  distance_metro,
    'age_years':       age_years,
    'locality_rating': locality_rating,
    'nearby_schools':  nearby_schools,   # correlated with locality
    'floor_number':    floor_number,
    'has_lift':        has_lift,          # correlated with floor
    'price_lakhs':     np.round(price_lakhs, 2)
})

print("Dataset shape:", df.shape)
print("\nCorrelation between correlated feature pairs:")
print(f"  area_sqft vs num_bedrooms  : {df['area_sqft'].corr(df['num_bedrooms']):.3f}")
print(f"  locality vs nearby_schools : {df['locality_rating'].corr(df['nearby_schools']):.3f}")
print(f"  floor vs has_lift          : {df['floor_number'].corr(df['has_lift']):.3f}")
print("\nHigh correlation = multicollinearity = Linear Regression will struggle")

In [ ]:
# ── Step 2: Split Data ────────────────────────────────────────────────────────
X = df.drop('price_lakhs', axis=1)
y = df['price_lakhs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# ── Step 3: Linear Regression — Witness the Wild Coefficients ─────────────────
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_pred  = lr_model.predict(X_test)
lr_rmse  = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2    = r2_score(y_test, lr_pred)

print("Linear Regression Coefficients:")
for feat, coef in zip(X.columns, lr_model.coef_):
    print(f"  {feat:<20}: {coef:>10.4f}")
print(f"\nRMSE : ₹{lr_rmse:.2f} lakhs")
print(f"R²   : {lr_r2:.4f}")
print("\nNotice: area_sqft and num_bedrooms coefficients fight each other")
print("        locality_rating and nearby_schools do the same")

In [ ]:
# ── Step 4: Ridge Regression — Calm, Stable Coefficients ─────────────────────

# WHY Pipeline?
# We MUST scale before Ridge. But we can only fit the scaler on training data.
# Pipeline ensures: scaler fits on X_train only, then transforms both X_train and X_test.
# Without Pipeline, you risk fitting the scaler on ALL data = data leakage.

ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Step 1: scale features (mean=0, std=1)
    ('ridge',  Ridge(alpha=1.0))   # Step 2: Ridge with alpha=1.0 (moderate penalty)
])
# WHY alpha=1.0 as starting point?
# It's sklearn's default — a reasonable moderate penalty.
# We'll find the optimal alpha properly using cross-validation below.

ridge_pipeline.fit(X_train, y_train)
ridge_pred = ridge_pipeline.predict(X_test)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2   = r2_score(y_test, ridge_pred)

# Extract Ridge coefficients (they're inside the pipeline)
ridge_coefs = ridge_pipeline.named_steps['ridge'].coef_

print("Ridge Regression Coefficients (alpha=1.0):")
for feat, coef in zip(X.columns, ridge_coefs):
    print(f"  {feat:<20}: {coef:>10.4f}")
print(f"\nRMSE : ₹{ridge_rmse:.2f} lakhs")
print(f"R²   : {ridge_r2:.4f}")
print("\nNotice: all coefficients are now calm and reasonable")

In [ ]:
# ── Step 5: Side-by-Side Coefficient Comparison ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Regression coefficients
axes[0].barh(X.columns, lr_model.coef_,
             color=['green' if c > 0 else 'red' for c in lr_model.coef_])
axes[0].axvline(x=0, color='black', linewidth=0.8)
axes[0].set_title('Linear Regression\n(Wild Coefficients from Multicollinearity)')
axes[0].set_xlabel('Coefficient Value')

# Ridge coefficients
axes[1].barh(X.columns, ridge_coefs,
             color=['green' if c > 0 else 'red' for c in ridge_coefs])
axes[1].axvline(x=0, color='black', linewidth=0.8)
axes[1].set_title('Ridge Regression (alpha=1.0)\n(Calm, Stable Coefficients)')
axes[1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.show()

# WHY this comparison plot?
# This is the core visual proof of what Ridge does:
# Left: correlated features fighting each other (huge +/- values)
# Right: Ridge forces all coefficients to be reasonable and stable

In [ ]:
# ── Step 6: How Alpha Controls Shrinkage ──────────────────────────────────────
# Let's see what happens to coefficients as we increase alpha

alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]
coef_paths = []  # store coefficients for each alpha

for alpha in alphas:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge',  Ridge(alpha=alpha))
    ])
    pipe.fit(X_train, y_train)
    coef_paths.append(pipe.named_steps['ridge'].coef_)

coef_paths = np.array(coef_paths)
# coef_paths shape: (8 alphas, 8 features)
# Each row = all feature coefficients at that alpha value

plt.figure(figsize=(10, 5))
for i, feat in enumerate(X.columns):
    plt.plot(np.log10(alphas), coef_paths[:, i],
             marker='o', label=feat, linewidth=1.5)

plt.xlabel('log₁₀(alpha)  →  stronger penalty →')
plt.ylabel('Coefficient value')
plt.title('Ridge Coefficient Paths — All Coefficients Shrink Toward Zero as Alpha Increases')
plt.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# WHY this plot?
# Shows the RIDGE PATH — how each coefficient shrinks as penalty increases.
# Key observations:
# - Small alpha (left): coefficients can be large and wild
# - Large alpha (right): all coefficients approach zero but NEVER reach it
# - No coefficient ever hits exactly zero (that would be Lasso)
print("Notice: no coefficient ever reaches exactly 0.0 — Ridge only shrinks, never eliminates")

In [ ]:
# ── Step 7: Find the Best Alpha with Cross-Validation ────────────────────────
# Instead of guessing alpha, we let cross-validation find the best one.

# RidgeCV does this automatically — it tries all given alphas using CV
# WHY cross-validation?
# We can't use the test set to pick alpha — that would be data leakage.
# Cross-validation uses only training data to estimate performance at each alpha.

alpha_candidates = [0.01, 0.1, 1, 10, 50, 100, 500, 1000]

ridge_cv_pipeline = Pipeline([
    ('scaler',   StandardScaler()),
    ('ridge_cv', RidgeCV(
        alphas=alpha_candidates,  # try all these
        cv=5,                     # 5-fold cross-validation
        scoring='neg_mean_squared_error'  # pick alpha with lowest MSE
    ))
])

ridge_cv_pipeline.fit(X_train, y_train)

best_alpha = ridge_cv_pipeline.named_steps['ridge_cv'].alpha_
# .alpha_ (with underscore) = the best alpha found after CV fitting

cv_pred = ridge_cv_pipeline.predict(X_test)
cv_rmse = np.sqrt(mean_squared_error(y_test, cv_pred))
cv_r2   = r2_score(y_test, cv_pred)

print(f"Best alpha found by cross-validation: {best_alpha}")
print(f"\nFinal Model Performance (best alpha):")
print(f"  RMSE : ₹{cv_rmse:.2f} lakhs")
print(f"  R²   : {cv_r2:.4f}")

In [ ]:
# ── Step 8: Full Comparison Table ─────────────────────────────────────────────
print("=" * 50)
print("MODEL COMPARISON — BANGALORE FLAT PRICE")
print("=" * 50)
print(f"{'Model':<30} {'RMSE':>8} {'R²':>8}")
print("-" * 50)
print(f"{'Linear Regression':<30} ₹{lr_rmse:>6.2f}L  {lr_r2:>6.4f}")
print(f"{'Ridge (alpha=1.0)':<30} ₹{ridge_rmse:>6.2f}L  {ridge_r2:>6.4f}")
print(f"{'Ridge (best alpha via CV)':<30} ₹{cv_rmse:>6.2f}L  {cv_r2:>6.4f}")
print("=" * 50)
print(f"\nBest alpha: {best_alpha}")
print("Ridge with CV-tuned alpha gives the most reliable results.")

In [ ]:
# ── Step 9: Visualise Final Predictions ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, cv_pred, alpha=0.4, color='steelblue', s=20)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price (₹ lakhs)')
axes[0].set_ylabel('Predicted Price (₹ lakhs)')
axes[0].set_title(f'Ridge (alpha={best_alpha}) — Actual vs Predicted')
axes[0].legend()

# Residuals
residuals = y_test - cv_pred
axes[1].scatter(cv_pred, residuals, alpha=0.4, color='tomato', s=20)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted Price (₹ lakhs)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot — Should Be Random Around Zero')

plt.tight_layout()
plt.show()

In [ ]:
# ── Step 10: From Scratch — Ridge Loss and Gradient Descent ──────────────────
# Manually implementing Ridge to show exactly how the penalty modifies training

# Single feature for simplicity: area_sqft → price
X_s = df['area_sqft'].values.astype(float)
y_s = df['price_lakhs'].values

# Scale manually
X_s = (X_s - X_s.mean()) / X_s.std()

def ridge_train(X, y, lam, lr=0.01, epochs=500):
    """Train Ridge Regression from scratch using gradient descent."""
    w, b = 0.0, 0.0
    n = len(X)
    loss_history = []

    for _ in range(epochs):
        y_hat = w * X + b

        # MSE part of loss
        mse = np.mean((y - y_hat) ** 2)

        # Ridge loss = MSE + lambda * w²
        # WHY only penalise w and not b?
        # The intercept b is not a feature weight — penalising it would
        # shift predictions up/down which makes no sense.
        ridge_loss = mse + lam * w**2
        loss_history.append(ridge_loss)

        # Gradients
        dw = (-2/n) * np.sum((y - y_hat) * X) + 2 * lam * w
        # WHY + 2*lam*w?
        # This is the derivative of the penalty term (lambda * w²) w.r.t. w
        # d/dw (lambda * w²) = 2 * lambda * w
        # This is the extra Ridge push — proportional to current w size

        db = (-2/n) * np.sum(y - y_hat)
        # No penalty term here — intercept is not penalised

        w = w - lr * dw
        b = b - lr * db

    return w, b, loss_history

# Compare different lambdas
lambdas = [0, 0.1, 1, 10]
plt.figure(figsize=(10, 4))

for lam in lambdas:
    w_final, b_final, losses = ridge_train(X_s, y_s, lam=lam)
    plt.plot(losses, label=f'λ={lam} → final w={w_final:.3f}')

plt.title('Ridge Loss During Training — Higher λ = Smaller Final Coefficient')
plt.xlabel('Epoch')
plt.ylabel('Ridge Loss')
plt.legend()
plt.tight_layout()
plt.show()

print("\nFinal coefficients at different lambda values:")
for lam in lambdas:
    w_f, _, _ = ridge_train(X_s, y_s, lam=lam)
    print(f"  λ={lam:<5} → w = {w_f:.4f}  {'(no penalty — pure LR)' if lam==0 else ''}")

---

## Summary Table

| | Ridge Regression |
|---|---|
| **Task** | Regression with many/correlated features |
| **Loss** | $MSE + \lambda \sum w_i^2$ |
| **Penalty type** | L2 — sum of squared coefficients |
| **Effect on coefficients** | Shrinks toward zero, never reaches zero |
| **Features eliminated?** | No — all features kept |
| **Key hyperparameter** | `alpha` (= λ) — controls penalty strength |
| **Must scale first?** | Yes — StandardScaler mandatory |
| **How to pick alpha** | RidgeCV with cross-validation |
| **Bias** | Slightly higher than Linear Regression |
| **Variance** | Lower — more stable, generalises better |
| **Strength** | Handles multicollinearity, prevents overfitting |
| **Weakness** | Cannot eliminate features entirely |
| **When to use** | Many features, correlated features, LR overfitting |

---

## What's Next?

Ridge shrinks all coefficients but **never to zero** — it keeps every feature.

But what if some features are completely useless? Ridge keeps them anyway (just very small).

**5.3 Lasso Regression** uses a different penalty (L1 — absolute value instead of squared) that **can** push coefficients to exactly zero — eliminating useless features entirely. This is called **automatic feature selection**.

---

## Practice Task

Suresh works at a bank in Chennai predicting **loan default risk score** (0–100) for applicants. Features include:
- `income_lakhs` — annual income
- `loan_amount_lakhs` — loan requested
- `credit_score` — CIBIL score (300–900)
- `existing_emis` — number of existing EMIs
- `loan_to_income` — loan_amount / income (correlated with both!)
- `debt_ratio` — existing_emis / income (correlated with existing_emis and income!)

**Your tasks:**

1. Create a synthetic dataset of 400 applicants with correlated features
2. Train Linear Regression — print and observe the wild coefficients
3. Train Ridge with alpha=1.0 — compare coefficients
4. Use RidgeCV to find the best alpha automatically
5. Compare RMSE of Linear Regression vs Ridge (best alpha)
6. Plot the coefficient path as alpha increases from 0.001 to 10,000

In [ ]:
# YOUR CODE HERE

# Step 1: Create dataset with correlated features

# Step 2: Linear Regression — observe wild coefficients

# Step 3: Ridge with alpha=1.0

# Step 4: RidgeCV — find best alpha

# Step 5: Compare RMSE

# Step 6: Coefficient path plot